## Prepare dictionary model for ET-OptME

In [1]:
import sys
%run  './script/pyomo_solving.py'
import os
import pandas as pd
import cobra
import json
import copy
from cobra import Model, Reaction, Metabolite

## Set path for models and data

In [2]:
# case study
model_file="./file/ecCZ870.json" # c.glutamicum model, irreversible and reactions with isoezyme splited
reaction_g0_file=os.path.join('./file/iCZ870_g0_n.csv') #  standard Gibbs free energy files
metabolites_lnC_file = os.path.join('file/iCZ870_met.txt') # concentration range files
reaction_kcat_MW_file= './file/reaction_kcat_MW_modify_n.csv'
# results
path_results = './results' # results saving path
dictionarymodel_path='./results/etcz870.json' # dictionary model saving path

## load and convert the model to dictionary model

In [3]:
model=cobra.io.load_json_model(model_file)
dictionary_model=trans_model2standard_json_etgem(model_file)
[reaction_list,metabolite_list,lb_list,ub_list,coef_matrix]=get_data(model)

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2788620
Academic license 2788620 - for non-commercial use only - registered to 90___@qq.com
./ecCZ870_irreversible.json


In [12]:
model.reactions.rxn00062_c

Reaction identifier,rxn00062_c
Name,ATP phosphohydrolase
Memory address,0x7e69a8f33b20
Stoichiometry,cpd00001_c + cpd00002_c --> cpd00008_c + cpd00009_c + cpd00067_c H2O + ATP --> ADP + Phosphate + H+
GPR,CEY17_RS06570 and CEY17_RS06580 and CEY17_RS06585 and CEY17_RS06590 and CEY17_RS06595 and...
Lower bound,0.8
Upper bound,1000.0


## get enzyme parameter

In [13]:
#get kcat_dict and mw_dict
kcat_mw=pd.read_csv(reaction_kcat_MW_file,index_col=0)
kcat_dict={}
mw_dict={}
for i in kcat_mw.index:
    if i in model.reactions:
        kcat_dict[i]=kcat_mw.loc[i,'kcat']
        enz=str(model.reactions.get_by_id(i).gpr)
        flag1=enz.split(' and ')
        flag1.sort(key=None, reverse=False)
        flag2=' and '.join(flag1)
        mw_dict[flag2]=kcat_mw.loc[i,'MW']

In [14]:
kcat_mw

,data_type,kcat,MW,kcat_MW
reactions,,,,
rxn05740_c_num1,DLKcat,1.507480e+07,15.456031,975334.650424
rxn05740_c_reverse_num1,DLKcat,1.233385e+07,15.456031,797995.648495
rxn00427_c,DLKcat,9.807317e+06,58.694084,167092.086944
Bio_protein_c,DLKcat,8.057041e+06,3504.117985,2299.306380
rxn00799_c,DLKcat,6.972236e+06,199.051125,35027.362578
...,...,...,...,...
rxn00330_c,modify,1.368063e+05,82.348781,1661.302923
rxn01115_c,modify,5.576734e+04,103.466874,538.987381
rxn00337_c,modify,5.799381e+04,44.738190,1296.293226


## Add kcat/mw to the dictionary model


In [15]:
def add_parameter(dictionarymodel,kcatdata,kmdata,mwdata):
    for enz in dictionarymodel['enzyme']:
        if enz in mwdata.keys():
            dictionarymodel['enzyme'][enz]['MW']=mwdata[enz]
        for rea in dictionarymodel['enzyme'][enz]['reactions']:
            if  rea in kcatdata.keys():
                dictionarymodel['enzyme'][enz]['reactions'][rea]['kcat']=kcatdata[rea]
            if  rea in kmdata.keys():
                dictionarymodel['enzyme'][enz]['reactions'][rea]['km']=kmdata[rea]
    for enz in mwdata:
        if enz not in dictionarymodel['enzyme']:
            print(enz)

## write the dictionary model to a file

In [16]:
add_parameter(dictionary_model,kcat_dict,{},mw_dict)
json_write(dictionarymodel_path,dictionary_model)